In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

import pandas as pd
from src.utils.db import get_connection

conn = get_connection()

games_df = pd.read_sql(
    "SELECT * FROM steam_indie_games",
    conn
)

reviews_df = pd.read_sql(
    "SELECT * FROM steam_indie_reviews",
    conn
)

summary_df = pd.read_sql(
    "SELECT * FROM steam_indie_review_summary",
    conn
)

histogram_df = pd.read_sql(
    "SELECT * FROM steam_indie_review_histogram",
    conn
)

tags_df = pd.read_sql(
    "SELECT * FROM steam_indie_tags",
    conn
)

conn.close()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28084\2074152488.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  games_df = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28084\2074152488.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews_df = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28084\2074152488.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  summary_df = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_28084\2074152488.py

In [2]:
games_df.shape

(9692, 14)

In [25]:
reviews_df.shape

(236379, 21)

In [28]:
summary_df.shape

(200, 7)

In [3]:
summary_list = summary_df['appid'].to_list()
reviews_list = reviews_df['appid'].unique().tolist()

In [4]:
diff_a = list(set(summary_list) - set(reviews_list))
diff_a

[2837370, 1818590, 2716270]

In [5]:
games_df[games_df['appid'].isin(diff_a)]

,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name
4002,2716270,"0 .. 20,000",18,2,0,0,"['Casual', 'Indie', 'Simulation']",2023-12-22,Harvey games,20,0,False,False,The Witch's Cauldron Prologue
7371,1818590,"0 .. 20,000",51,29,99,0,"['Action', 'Adventure', 'Indie', 'Racing']",2024-01-06,GUNTER GAMES,80,0,False,False,You Are A Pilot
9391,2837370,"0 .. 20,000",10,7,399,0,"['Action', 'Indie']",2024-02-27,StarSystemStudios,17,0,False,False,NeverGoingHome


In [5]:
reviews_df[reviews_df['appid'].isin(diff_a)]

,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,steam_purchase,received_for_free,written_during_early_access,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played


In [13]:
#games_df['is_f2p'] = games_df['is_f2p'].str.lower().replace({'true' : True, 'false' : False})
#games_df['is_early_access'] = games_df['is_early_access'].str.lower().replace({'true' : True, 'false' : False})
games_df['is_early_access'].value_counts()

is_early_access
False    9692
Name: count, dtype: int64

In [2]:
games_df['release_date'] = pd.to_datetime(games_df['release_date'])
games_df['release_date'].dtypes

dtype('<M8[us]')

In [3]:
games_df['price'] = games_df['price'].apply(pd.to_numeric, errors = 'coerce')
games_df['price'].isna().sum()

np.int64(0)

In [4]:
games_df['price'] = games_df['price'] / 100

In [5]:
games_df = games_df.drop(columns = ['is_f2p', 'is_early_access'])
games_df.dtypes

appid                     int64
owners                      str
positive                  int64
negative                  int64
price                   float64
ccu                         str
genres                      str
release_date     datetime64[us]
developers                  str
total_reviews               str
owners_lower                str
name                        str
dtype: object

In [6]:
games_df['total_reviews'] = games_df['total_reviews'].apply(pd.to_numeric, errors = 'coerce')
games_df['total_reviews'].isna().sum()

np.int64(0)

In [7]:
games_df['owners_lower'] = games_df['owners_lower'].apply(pd.to_numeric, errors = 'coerce')
games_df['owners_lower'].isna().sum()

np.int64(0)

In [8]:
import ast
games_df['genres'] = games_df['genres'].apply(ast.literal_eval)
games_df.dtypes

appid                     int64
owners                      str
positive                  int64
negative                  int64
price                   float64
ccu                         str
genres                   object
release_date     datetime64[us]
developers                  str
total_reviews             int64
owners_lower              int64
name                        str
dtype: object

In [9]:
type(games_df[games_df['name'] == '7 Days to Die']['genres'].item())

list

In [11]:
games_df['owners'] = games_df['owners'].str.replace(',', '', regex = False)
games_df['owners_higher'] = games_df['owners'].str.split(r'\.\.').str[1].str.strip()
games_df['owners_higher'] = pd.to_numeric(games_df['owners_higher'], errors = 'coerce')
games_df = games_df.drop(columns = ['owners', 'ccu'])
games_df.dtypes

appid                     int64
positive                  int64
negative                  int64
price                   float64
genres                   object
release_date     datetime64[us]
developers                  str
total_reviews             int64
owners_lower              int64
name                        str
owners_higher             int64
dtype: object